# Feed data to S3

The goal of this short exercise is to practise using `boto3` to communicate with the Amazon Data Lake service: S3.

The exercise is organized in two parts:
* Part one will have you upload a file from your local system directly to s3
* Part two will show you how to send files to your s3 that only exist as local python variables

## Part 1

1. Start by installing `boto3` if it has not been done already

In [147]:
#!pip install -q xgboost
#!pip install -q s3fs
 #!pip install -U kaleido
!pip install Boto3
#!pip install s3fs
# Load in our libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
# import s3fs
import boto3

# plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.figure_factory as ff
import plotly.colors as pc

# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "colab" # pour que colab ne bloque pas l'export svg

import warnings
warnings.filterwarnings('ignore')



# montage drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [148]:
# Install boto3 using pip
## Add '!' only if you install directly from a Jupyter Notebook


2. Import the necessary libraries

3. Create an instance of `boto3.Session` that connects with your aws account.

In [149]:
from google.colab import userdata
# sur vs code utiliser aws configure
#
#aws configure

#
print(userdata.get("AWS_ACCESS_KEY_ID") is not None)
print(userdata.get("AWS_SECRET_ACCESS_KEY") is not None)


True
True


| AWS            | Analogie entreprise                                     |
| -------------- | ------------------------------------------------------- |
| **`boto3`**    | Le **plan du site** / le **répertoire de l’entreprise** |
| **`Session`**  | Ton **badge nominatif** (identité + droits + site)      |
| **`client`**   | Le **guichet d’un service** (accueil, IT, logistique)   |
| **`resource`** | L’**accès interne au service** (open space, étages)     |
| **Bucket S3**  | Une **salle d’archives / entrepôt de documents**        |
| Object S3      | Un **dossier ou fichier** dans l’archive                |


Tu peux entrer sur le site (boto3)
Mais sans badge (Session), personne ne te laisse entrer dans les bureaux

In [150]:
region = userdata.get("AWS_DEFAULT_REGION") or "eu-west-1"
# boto3 est une bibliothèque.
# Session est une instance de connexion AWS.

session = boto3.Session(
    aws_access_key_id=userdata.get("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=userdata.get("AWS_SECRET_ACCESS_KEY"),
    # region_name=region
)


| Action              | client       | resource    |
| ------------------- | ------------ | ----------- |
| Créer un bucket     | ✅ recommandé | ⚠️ possible |
| Lister les buckets  | ✅            | ❌           |
| Gérer les erreurs   | ✅            | ⚠️          |
| Manipuler un bucket | ❌            | ✅           |
| Upload / download   | ⚠️           | ✅           |


In [151]:

from botocore.exceptions import ClientError

s3_client = session.client("s3")
bucket_name = "olivier-jedha-datalake-2026"

def bucket_exists(bucket_name):
    try:
        s3_client.head_bucket(Bucket=bucket_name)
        return True
    except ClientError:
        return False
# s3 = .resource("s3")
# s3.create_bucket()
if not bucket_exists(bucket_name):
    s3_client.create_bucket(
        Bucket=bucket_name,
        CreateBucketConfiguration={
            "LocationConstraint": "us-west-1"
        }
    )
    print("Bucket créé")
else:
    print("Bucket déjà existant")


Bucket déjà existant


# pourquoi un contrôle de flux par exception dans le cloud ?

Si je redemande à AWS de créer un bucket qui existe déjà, AWS doit répondre quelque chose.
premier run -> création du bucket
2e run -> BucketAlreadyOwnedByYou -> le script plante en production

principe d'Idempotence!!!:
--> Je peux exécuter ce code 1 fois, 10 fois ou 100 fois,
et j’obtiens toujours le même résultat sans erreur.


4. Create a variable called `s3` that connects your session to the s3 ressource.

In [152]:
s3_resource = session.resource("s3")


5. Create a variable called `bucket` that will connect to an existing bucket in your s3 or to a bucket you are creating now.

In [153]:
bucket = s3_resource.Bucket(bucket_name)



In [154]:
bucket.creation_date


datetime.datetime(2026, 1, 5, 3, 8, 36, tzinfo=tzlocal())

6. In the [documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3.html#S3.Bucket.upload_file), find a method that lets you upload files to your s3, and upload the file `crypto.json` to your bucket in the folder of your choice.

In [155]:
bucket.upload_file( #
    Filename="/content/drive/MyDrive/JedhaBootcamp/crypto_data.json",
    Key="raw/crypto.json" # je lui donne le nom que je veux dans S3 dans un dossier raw/
)


In [156]:
s3_client.list_buckets()


{'ResponseMetadata': {'RequestId': 'PNH3HPAW8RSNRAZQ',
  'HostId': 'gv4y+yBGPEYCZwHOsaEJTtu/upKJMhmUjY8tpVe8T3SalGttVYTAuZvakNnbWy/1Z9zcTHyAjEWm0iBQoiuzAClo/LRYFi/tqNKbdjKtHlc=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'gv4y+yBGPEYCZwHOsaEJTtu/upKJMhmUjY8tpVe8T3SalGttVYTAuZvakNnbWy/1Z9zcTHyAjEWm0iBQoiuzAClo/LRYFi/tqNKbdjKtHlc=',
   'x-amz-request-id': 'PNH3HPAW8RSNRAZQ',
   'date': 'Thu, 08 Jan 2026 16:10:53 GMT',
   'content-type': 'application/xml',
   'transfer-encoding': 'chunked',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'Buckets': [{'Name': 'olivier-jedha-datalake-2026',
   'CreationDate': datetime.datetime(2026, 1, 5, 3, 8, 36, tzinfo=tzlocal()),
   'BucketArn': 'arn:aws:s3:::olivier-jedha-datalake-2026'}],
 'Owner': {'ID': 'a7a691a2875e500ab7896c3075735d387b68862bb7cabf4face801b294879b99'}}

## Part 2

1. Using the `json` library, load the `crypto_data.json` file in a variable called `crypto`

In [157]:

with open("/content/drive/MyDrive/JedhaBootcamp/crypto_data.json", "r") as f:
    crypto = json.load(f)


2. What is the type of variable `crypto`

In [158]:
type(crypto)


dict

3. Display the different possible keys in `crypto` in the form of a list

In [159]:
list(crypto.keys())


['btc-bitcoin',
 'eth-ethereum',
 'usdt-tether',
 'bnb-binance-coin',
 'hex-hex',
 'usdc-usd-coin',
 'ada-cardano',
 'sol-solana',
 'xrp-xrp',
 'luna-terra']

# Rappel : pourquoi convertir en liste et ne pas manipuler directement le dictionnaire crypto ?
crypto["btc-bitcoin"]  # OK
crypto[0]              # KO, impossible d'indexer le premier élément, d'accéder à ses valeurs
crypto.keys.[0]              # KO -> ne marchera pas non plus car TypeError


# on convertit donc en liste pour rendre les clés indexables
-> ce qui donne tous les éléments de la liste en clés-valeurs.

exception en cas de boucle

In [160]:
for key, values in crypto.items():
    print(key, values[:2])


btc-bitcoin [{'time_open': '2012-01-01T00:00:00Z', 'time_close': '2012-01-01T23:59:59Z', 'open': 4.72, 'high': 4.72, 'low': 4.72, 'close': 4.72}, {'time_open': '2012-01-02T00:00:00Z', 'time_close': '2012-01-02T23:59:59Z', 'open': 5.27, 'high': 5.27, 'low': 5.27, 'close': 5.27}]
eth-ethereum [{'time_open': '2015-08-07T00:00:00Z', 'time_close': '2015-08-07T23:59:59Z', 'open': 2.83162, 'high': 3.53661, 'low': 2.52112, 'close': 2.77212, 'volume': 164329}, {'time_open': '2015-08-08T00:00:00Z', 'time_close': '2015-08-08T23:59:59Z', 'open': 2.79376, 'high': 2.79881, 'low': 0.714725, 'close': 0.753325, 'volume': 674188, 'market_cap': 167911166}]
usdt-tether [{'time_open': '2015-02-25T00:00:00Z', 'time_close': '2015-02-25T23:59:59Z', 'open': 1.21016, 'high': 1.21549, 'low': 1.20958, 'close': 1.2111, 'volume': 5, 'market_cap': 304476}, {'time_open': '2015-02-26T00:00:00Z', 'time_close': '2015-02-26T23:59:59Z', 'open': 1.21042, 'high': 1.21232, 'low': 1.19471, 'close': 1.20574, 'volume': 5, 'mark

In [161]:
it = iter(crypto)
first_key = next(it)
first_key



'btc-bitcoin'

In [162]:
second_key = next(it)
second_key

'eth-ethereum'

In [163]:
third_key = next(it)
third_key

'usdt-tether'

4. Have a look at the value associated with the first key, do you think it could easily be converted to a pandas DataFrame?

In [164]:
first_key = list(crypto.keys())[0]
first_key


'btc-bitcoin'

#rappel :
crypto = {
    "btc-bitcoin": [...],
    "eth-ethereum": [...],
    "usdt-tether": [...],
}


In [182]:
values = crypto[first_key] # donne moi la valeur associée à la premi!re clé du dictionnaire crypto
# value = dictionnaire [clé]

values[:2] # values = liste de dictionnaires

[{'time_open': '2012-01-01T00:00:00Z',
  'time_close': '2012-01-01T23:59:59Z',
  'open': 4.72,
  'high': 4.72,
  'low': 4.72,
  'close': 4.72},
 {'time_open': '2012-01-02T00:00:00Z',
  'time_close': '2012-01-02T23:59:59Z',
  'open': 5.27,
  'high': 5.27,
  'low': 5.27,
  'close': 5.27}]

In [183]:
first_record = values[31]
first_record

{'time_open': '2012-02-01T00:00:00Z',
 'time_close': '2012-02-01T23:59:59Z',
 'open': 5.48,
 'high': 5.48,
 'low': 5.48,
 'close': 5.48}

La valeur associée à chaque clé est une liste de dictionnaires homogènes, ce qui correspond naturellement à des lignes et colonnes dans un DataFrame pandas.

5. Convert the previous object to a pandas dataframe

In [167]:
df_btc = pd.DataFrame(crypto["btc-bitcoin"])
df_btc.head()

,time_open,time_close,open,high,low,close,market_cap,volume
0,2012-01-01T00:00:00Z,2012-01-01T23:59:59Z,4.72,4.72,4.72,4.72,NaN,NaN
1,2012-01-02T00:00:00Z,2012-01-02T23:59:59Z,5.27,5.27,5.27,5.27,NaN,NaN
2,2012-01-03T00:00:00Z,2012-01-03T23:59:59Z,5.22,5.22,5.22,5.22,NaN,NaN
3,2012-01-04T00:00:00Z,2012-01-04T23:59:59Z,4.88,4.88,4.88,4.88,NaN,NaN
4,2012-01-05T00:00:00Z,2012-01-05T23:59:59Z,5.57,5.57,5.57,5.57,NaN,NaN


6. Now that you know how to convert one element of `crypto` into a dataframe, write ONE line of code that will convert each element into a dataframe and store them in a `list` object.

<details>
  <summary>Spoiler</summary>
  Use a list comprehension.
</details>


In [187]:
dfs = [pd.DataFrame(v) for v in crypto.values()]
# dfs.head() # list object has no attribute head!
dfs[0].head()

,time_open,time_close,open,high,low,close,market_cap,volume
0,2012-01-01T00:00:00Z,2012-01-01T23:59:59Z,4.72,4.72,4.72,4.72,NaN,NaN
1,2012-01-02T00:00:00Z,2012-01-02T23:59:59Z,5.27,5.27,5.27,5.27,NaN,NaN
2,2012-01-03T00:00:00Z,2012-01-03T23:59:59Z,5.22,5.22,5.22,5.22,NaN,NaN
3,2012-01-04T00:00:00Z,2012-01-04T23:59:59Z,4.88,4.88,4.88,4.88,NaN,NaN
4,2012-01-05T00:00:00Z,2012-01-05T23:59:59Z,5.57,5.57,5.57,5.57,NaN,NaN


7. For each of these dataframes add a column called `"coin"` that contains the name of the key it was associated with.

<details>
  <summary>Spoiler</summary>
  Use a for loop and the `zip` command to be able to loop over two iterables at once
</details>

In [192]:
dfs = [
    pd.DataFrame(v).assign(coin=k)
    for k, v in crypto.items()
]
dfs[1].head()

,time_open,time_close,open,high,low,close,volume,market_cap,coin
0,2015-08-07T00:00:00Z,2015-08-07T23:59:59Z,2.831620,3.536610,2.521120,2.772120,164329,NaN,eth-ethereum
1,2015-08-08T00:00:00Z,2015-08-08T23:59:59Z,2.793760,2.798810,0.714725,0.753325,674188,167911166.0,eth-ethereum
2,2015-08-09T00:00:00Z,2015-08-09T23:59:59Z,0.706136,0.879810,0.629191,0.701897,532170,42637551.0,eth-ethereum
3,2015-08-10T00:00:00Z,2015-08-10T23:59:59Z,0.713989,0.729854,0.636546,0.708448,405283,43130016.0,eth-ethereum
4,2015-08-11T00:00:00Z,2015-08-11T23:59:59Z,0.708087,1.131410,0.663235,1.067860,1463100,42796545.0,eth-ethereum


équivalent zip trop verbeux:

In [170]:
keys = crypto.keys()
values = crypto.values()


In [171]:
df_zip = []

for k, v in zip(keys, values):
    df = pd.DataFrame(v)
    df["coin"] = k
    df_zip.append(df)


In [189]:
df_zip[1].head()

,time_open,time_close,open,high,low,close,volume,market_cap,coin
0,2015-08-07T00:00:00Z,2015-08-07T23:59:59Z,2.831620,3.536610,2.521120,2.772120,164329,NaN,eth-ethereum
1,2015-08-08T00:00:00Z,2015-08-08T23:59:59Z,2.793760,2.798810,0.714725,0.753325,674188,167911166.0,eth-ethereum
2,2015-08-09T00:00:00Z,2015-08-09T23:59:59Z,0.706136,0.879810,0.629191,0.701897,532170,42637551.0,eth-ethereum
3,2015-08-10T00:00:00Z,2015-08-10T23:59:59Z,0.713989,0.729854,0.636546,0.708448,405283,43130016.0,eth-ethereum
4,2015-08-11T00:00:00Z,2015-08-11T23:59:59Z,0.708087,1.131410,0.663235,1.067860,1463100,42796545.0,eth-ethereum


8. Upload each of these dataframes to the folder of your choice in your s3

<details>
  <summary>Spoiler</summary>
  Use the pandas method `to_csv` to convert your dataframes to csv files and the appropriate command from the bucket object
</details>

In [193]:
bucket_name = "olivier-jedha-datalake-2026"
bucket = s3_resource.Bucket(bucket_name)


In [194]:
for df in dfs:
    coin_name = df["coin"].iloc[0]
    s3_key = f"processed/{coin_name}.csv"

    csv_buffer = df.to_csv(index=False)

    s3_client.put_object(
        Bucket=bucket_name,
        Key=s3_key,
        Body=csv_buffer
    )

    print(f"Upload effectué : {s3_key}")


Upload effectué : processed/btc-bitcoin.csv
Upload effectué : processed/eth-ethereum.csv
Upload effectué : processed/usdt-tether.csv
Upload effectué : processed/bnb-binance-coin.csv
Upload effectué : processed/hex-hex.csv
Upload effectué : processed/usdc-usd-coin.csv
Upload effectué : processed/ada-cardano.csv
Upload effectué : processed/sol-solana.csv
Upload effectué : processed/xrp-xrp.csv
Upload effectué : processed/luna-terra.csv


Deuxième exécution

Les fichiers existent déjà

put_object ÉCRASE silencieusement les objets

Aucun message d’erreur

Les données sont remplacées

---------------------->   C’est dangereux :

historique perdu

on ne sais pas qu’un écrasement a eu lieu

impossible de distinguer un premier run d’un second

car S3 ne lève PAS d’erreur sur overwrite.

Donc :

pas de signal

pas de protection

pas d’audit simple


penser à un try except + head_object()

In [175]:
from botocore.exceptions import ClientError

def object_exists(bucket_name, key):
    try:
        s3_client.head_object(Bucket=bucket_name, Key=key)
        return True
    except ClientError:
        return False


In [176]:
for df in dfs:
    coin_name = df["coin"].iloc[0]
    s3_key = f"processed/{coin_name}.csv"

    if not object_exists(bucket_name, s3_key):
        csv_buffer = df.to_csv(index=False)

        s3_client.put_object(
            Bucket=bucket_name,
            Key=s3_key,
            Body=csv_buffer
        )
        print(f"Upload effectué : {s3_key}")
    else:
        print(f"Déjà présent, upload ignoré : {s3_key}")


Déjà présent, upload ignoré : processed/btc-bitcoin.csv
Déjà présent, upload ignoré : processed/eth-ethereum.csv
Déjà présent, upload ignoré : processed/usdt-tether.csv
Déjà présent, upload ignoré : processed/bnb-binance-coin.csv
Déjà présent, upload ignoré : processed/hex-hex.csv
Déjà présent, upload ignoré : processed/usdc-usd-coin.csv
Déjà présent, upload ignoré : processed/ada-cardano.csv
Déjà présent, upload ignoré : processed/sol-solana.csv
Déjà présent, upload ignoré : processed/xrp-xrp.csv
Déjà présent, upload ignoré : processed/luna-terra.csv


9. Now go to your S3 to make sure the files have been uploaded correctly!

In [195]:

[b["Name"] for b in s3_client.list_buckets()["Buckets"]]


['olivier-jedha-datalake-2026']

In [196]:
[obj.key for obj in bucket.objects.all()]

['processed/ada-cardano.csv',
 'processed/bnb-binance-coin.csv',
 'processed/btc-bitcoin.csv',
 'processed/eth-ethereum.csv',
 'processed/hex-hex.csv',
 'processed/luna-terra.csv',
 'processed/sol-solana.csv',
 'processed/usdc-usd-coin.csv',
 'processed/usdt-tether.csv',
 'processed/xrp-xrp.csv',
 'raw/crypto.json']

In [197]:

obj = s3_client.get_object(
    Bucket="olivier-jedha-datalake-2026",
    Key="processed/sol-solana.csv"
)

df_test = pd.read_csv(obj["Body"])
df_test.head()

,time_open,time_close,open,high,low,close,volume,market_cap,coin
0,2020-08-26T00:00:00Z,2020-08-26T23:59:59Z,3.614188,3.790380,3.453380,3.558723,18828792,114706133,sol-solana
1,2020-08-27T00:00:00Z,2020-08-27T23:59:59Z,3.558479,3.854475,3.298325,3.436928,17983302,113827454,sol-solana
2,2020-08-28T00:00:00Z,2020-08-28T23:59:59Z,3.457846,4.304559,3.457846,4.022359,23514694,111472486,sol-solana
3,2020-08-29T00:00:00Z,2020-08-29T23:59:59Z,4.015835,4.102584,3.792236,3.977288,12079357,130229641,sol-solana
4,2020-08-30T00:00:00Z,2020-08-30T23:59:59Z,3.965328,4.688419,3.915448,4.430499,19517464,128591246,sol-solana


In [180]:
s3_client.put_bucket_versioning(
    Bucket=bucket_name,
    VersioningConfiguration={
        "Status": "Enabled"
    }
)
s3_client.get_bucket_versioning(Bucket=bucket_name)


{'ResponseMetadata': {'RequestId': 'R8MGZ6N60RDYANDF',
  'HostId': 'IUDuXooRPwFJE7uE5Wfj04/Obapv6nItbM+jNiKqaapr4NXmtMxfLYUbSq/M21bUAfgPpdQ9Dg4=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'IUDuXooRPwFJE7uE5Wfj04/Obapv6nItbM+jNiKqaapr4NXmtMxfLYUbSq/M21bUAfgPpdQ9Dg4=',
   'x-amz-request-id': 'R8MGZ6N60RDYANDF',
   'date': 'Thu, 08 Jan 2026 16:11:01 GMT',
   'transfer-encoding': 'chunked',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'Status': 'Enabled'}